# Pre-processing river water surface elevation (WSE) files from Hydroweb.next to prepare for Altimetric Rating Curves (ARC) fitting
## Tranform the HydroWeb.next virtual station .txt data into a global .csv

_Notebook Authors: Arnaud Cerbelaud, NASA Jet Propulsion Laboratory - California Institute of Technology (April 2024 - April 2025)._ All rights reserved.

In [3]:
#!/usr/bin/env python3
# ******************************************************************************
# ARC_preprocess_Hydroweb.ipynb
# ******************************************************************************

# Purpose:
# Pre-processing water surface elevation files from Hydroweb.next
# Author:
# Arnaud Cerbelaud, 2026

In [2]:
# ******************************************************************************
# Import Python modules
# ******************************************************************************
import contextily as cx
import glob
import os
import requests
import pandas as pd
import numpy as np
from pathlib import Path
import zipfile

/Users/cerbelaud/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [ ]:
# ******************************************************************************
# Declaration of variables
# ******************************************************************************
# 1 - data_folder

######
# 1 - data_folder
######
data_folder = ""

# To adjust depending on the files downloaded... Numbers of header rows
n_col = 44

In [ ]:
# Example file, DO NOT RUN

#BASIN:: AAKOL
#RIVER:: EMEL
#ID:: 0000000009722
#TRIBUTARY OF:: NA
#APPROX. WIDTH OF REACH (m):: 50
#SURFACE OF UPSTREAM WATERSHED (km2):: NA
#RATING CURVE PARAMETERS A,b,Zo such that Q(m3/s) = A[H(m)-Zo]^b:: NA NA NA
#REFERENCE ELLIPSOID:: WGS84
#REFERENCE LONGITUDE:: 82.2061
#REFERENCE LATITUDE:: 46.3663
#REFERENCE DISTANCE (km):: 63
#GEOID MODEL:: EGM2008
#GEOID ONDULATION AT REF POSITION(M.mm):: -52.30
#MISSION(S)-TRACK(S):: S3B-0735
#STATUS:: OPERATIONAL
#VALIDATION CRITERIA:: AUTOMATIC
#MEAN ALTITUDE(M.mm):: 362.52
#MEAN SLOPE (mm/km):: NA
#NUMBER OF MEASUREMENTS IN DATASET:: 69
#FIRST DATE IN DATASET:: 2018-12-15
#LAST DATE IN DATASET:: 2024-11-13
#DISTANCE MIN IN DATASET (km):: 62.4
#DISTANCE MAX IN DATASET (km):: 64.0
#PRODUCTION DATE:: 2024-11-16
#PRODUCT VERSION:: 2.0
#PRODUCT CITATION:: DOI : https://doi.org/10.24400/329360/HYDROWEB_WATER_LEVEL
#SOURCES::
#PRODUCT CONTENT::
#COL 1 : DATE(YYYY-MM-DD)
#COL 2 : TIME(HH:MM)
#COL 3 : ORTHOMETRIC HEIGHT (M) OF WATER SURFACE AT REFERENCE POSITION
#COL 4 : ASSOCIATED UNCERTAINTY(M)
#FIELD SEPARATOR :
#COL 5 : LONGITUDE OF ALTIMETRY MEASUREMENT (deg)
#COL 6 : LATITUDE OF ALTIMETRY MEASUREMENT (deg)
#COL 7 : ELLIPSOIDAL HEIGHT OF ALTIMETRY MEASUREMENT (M)
#COL 8 : GEOIDAL ONDULATION (M) at location [5,6]
#COL 9 : DISTANCE OF ALTIMETRY MEASUREMENT TO REFERENCE POSITION(KM)
#COL 10 : SATELLITE
#COL 11 : ORBIT / MISSION
#COL 12 : GROUND-TRACK NUMBER
#COL 13 : CYCLE NUMBER
#COL 14 : RETRACKING ALGORITHM
#COL 15 : GDR VERSION
################################################################
2018-12-15 15:54 362.72 0.11 : 9999.999 9999.999 310.42 -52.30 9999.999 S3B REP 0735 019 OCOG NA
2019-03-06 15:54 362.49 0.11 : 9999.999 9999.999 310.20 -52.30 9999.999 S3B REP 0735 022 OCOG NA
...

In [5]:
# ******************************************************************************
# Read files
# ******************************************************************************
print('Reading files')

alti_datafiles = []
# Files are actually inside single-file folders
for foldername in os.listdir(data_folder):
    if foldername!='.DS_Store':
        try:
            alti_datafiles.append( data_folder + foldername + '/' +
                               os.listdir(data_folder + foldername)[0]
                             )
        except:
            alti_datafiles.append( data_folder + foldername
                             )

In [7]:
# Initializing dataframes
alti_data   = pd.DataFrame()
alti_data_u = pd.DataFrame()

n=0
for alti_file in alti_datafiles:
    # Program to read all the lines as a list in a file
    # using readlines() function
    alti    = open(alti_file, "r")
    content = alti.readlines()
    alti.close()
    
    # Get all the header informations (name of river, mission, geoid model, etc.)
    i=0
    while content[i][:3] != '###':  # before reaching the actual WSE data
        row_name   = content[i].partition("::")[0][1:]
        row_value  = content[i].partition("::")[2][1:-1]
        alti_data.loc[n,row_name]   = row_value  # header rows on VS
        alti_data_u.loc[n,row_name] = row_value  # header rows on VS
        i=i+1
    
    # Get all wse and wse_u data at each date (forget the hour)
    j=i+1
    while j < len(content):
        row_name   = content[j].partition(" ")[0] # this is the date of acquisition, I didn't keep the
        # time of day (which can be useful for tide correction though)
        wse   = content[j].partition(" ")[2].partition(" ")[2].partition(" ")[0]
        wse_u = content[j].partition(" ")[2].partition(" ")[2].partition(" ")[2].partition(" ")[0]
        alti_data.loc[n,row_name]   = np.float32(wse)
        alti_data_u.loc[n,row_name] = np.float32(wse_u)
        j=j+1
        
    n = n+1

/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data.loc[n,row_name]   = np.float32(wse)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data_u.loc[n,row_name] = np.float32(wse_u)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.i

/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data.loc[n,row_name]   = np.float32(wse)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data_u.loc[n,row_name] = np.float32(wse_u)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.i

/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data_u.loc[n,row_name] = np.float32(wse_u)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data.loc[n,row_name]   = np.float32(wse)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.i

/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data.loc[n,row_name]   = np.float32(wse)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data_u.loc[n,row_name] = np.float32(wse_u)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.i

/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data_u.loc[n,row_name] = np.float32(wse_u)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data.loc[n,row_name]   = np.float32(wse)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.i

...

/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data.loc[n,row_name]   = np.float32(wse)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data_u.loc[n,row_name] = np.float32(wse_u)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.i

/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data_u.loc[n,row_name] = np.float32(wse_u)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data.loc[n,row_name]   = np.float32(wse)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.i

/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data.loc[n,row_name]   = np.float32(wse)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data_u.loc[n,row_name] = np.float32(wse_u)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.i

/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data.loc[n,row_name]   = np.float32(wse)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data_u.loc[n,row_name] = np.float32(wse_u)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.i

/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data.loc[n,row_name]   = np.float32(wse)
/var/folders/b1/jf1jt9ys40l147sxl2g3js9r0000gr/T/ipykernel_94654/1172830389.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  alti_data_u.loc[n,row_name] = np.float32(wse_u)


In [8]:
alti_data   = pd.concat([alti_data.iloc[:,:n_col],  alti_data.iloc[:,n_col:].sort_index(axis = 1)],   axis = 1)
alti_data_u = pd.concat([alti_data_u.iloc[:,:n_col],alti_data_u.iloc[:,n_col:].sort_index(axis = 1)], axis = 1)


In [9]:
### Write to excel

alti_data.iloc[:,:n_col].to_csv(  data_folder + '_vs_database.csv')
alti_data.to_csv(                 data_folder + '_wse_database.csv')
alti_data_u.to_csv(               data_folder + '_wse_u_database.csv')
